In [1]:
import json
import typing
import ollama
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.runnables.graph import MermaidDrawMethod
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.graph import StateGraph, END
from pydantic import BaseModel 
from pprint import pprint
from IPython.display import Image, display

## Defining the open-source LLM with Ollama

In [2]:
llm_opensource = "llama3.2"

In [3]:
q = "Who are the top 10 most winners of the Premier League?"

q = "Who are the top 10 latest winners of Premier League?"

In [5]:
result = ollama.chat(model = llm_opensource, 
                        messages = [{"role":"system", "content":""},
                                    {"role":"user", "content":q}])

In [6]:
print(result.message.content)

Here are the top 10 latest winners of the Premier League, starting from the 2011-12 season:

1. 2012-13 - Manchester United
2. 2013-14 - Manchester City
3. 2014-15 - Chelsea
4. 2015-16 - Leicester City
5. 2016-17 - Chelsea
6. 2017-18 - Manchester City
7. 2018-19 - Manchester City
8. 2019-20 - Liverpool
9. 2020-21 - Manchester City
10. 2021-22 - Manchester City

Please note that the data is accurate as of the end of the 2021-22 season and may change in future seasons.


## Defining the Tools

In the LangGraph context, tools are external functions that the graph nodes can call to execute specific tasks, such as search in the web, interact with APIs and manipulate data. They allow the LangGraph conversation flow access external funcionalities, becoming the agent more dynamical and able to handle with different kind of requests.

In [7]:
@tool("search_tool")
def search_tool(q:str) -> str:
    """Search on DuckDuckGo navigator using `q` as input"""
    return DuckDuckGoSearchRun().run(q)

In [8]:
# Testing
print(search_tool.invoke(q))

1 month ago -Only five clubs have won the title in three consecutive seasons:Huddersfield Town, Arsenal, Liverpool, Manchester United (twice) and Manchester City, with the latter being the only club to have won four successive titles. ... 24 clubs which have won the English top level title, including 7 ... 1 day ago -Twenty clubs are competing in the 2025–26 season – top seventeen from the previous season and three promoted from the Championship. Leicester City, Ipswich Town and Southampton were relegated to the EFL Championship for the 2025–26 season, whilstLeeds United, Burnley, and Sunderland, as ... 1 week ago -Premier League · 20 20 13 10 9 7 6 6 4 4 3 3 3 3 2 2 2 2 2 1 1 1 1 1 · Tournament records Wins: Liverpool 20 time(s) Matches: J. Milner · J. Milner 657 games Goals: A. Shearer · A. Shearer 260 goals · €12.58bn · Total market value · Type of cup: First Tier · 1 week ago -Check the updated ranking of all the winners and runners-up of the English Premier League year by year sin

In [9]:
@tool("final_answer_tool")
def final_answer_tool(text:str) -> str:
    """
    Return a natural language answer to user using the `input text`.
    You must provide the maximum of possible context and specify the source of the information.
    """

    return text

In [10]:
# Testing
print(final_answer_tool.invoke("Test") )

Test


In [11]:
# Tools dict
dic_tools = {"search_tool":search_tool, 
             "final_answer_tool":final_answer_tool}

## Adding tools to help the decision-making processo of the Agent

In [12]:
prompt = """
You are a specialist in news, you must answer all the questions of the user, you can use the tools list provided to you.
Your goal is provide the best possible answer to the user, including important informations about the sources and tools used.

Look, when you call a tool, you need to provide the name of the tool and the argument that it uses in JSON format.
For each call, you MUST use ONLY on tool and the answer format ALWAYS NEEDS to follow this patter:
```json
{"name": "<tool_name>", "parameters": "<tool_input_key>":"<tool_input_value>"}
```

You must remember, You MUST NOT use the tool with the same query more than once.
You must remember, if the user doesn't make a specific question, you MUST use the tool `final_answer_tool` directly.

Everytime the user ask you a question, you need to store in your memory.
Everytime you find some information related to user question, you have to store in your memory.

You must try collect informations from several sources before providing a answer to the user.
After collect enough informations to answer the user question, use the tool `final_answer_tool`.
"""

In [13]:
str_tools = "\n".join([str(n+1)+". `"+str(v.name)+"`: "+str(v.description) for n,v in enumerate(dic_tools.values())])

In [14]:
print(str_tools)

1. `search_tool`: Search on DuckDuckGo navigator using `q` as input
2. `final_answer_tool`: Return a natural language answer to user using the `input text`.
You must provide the maximum of possible context and specify the source of the information.


In [15]:
# Tools prompt
prompt_tools = f"You can use the following tools:\n{str_tools}"

print(prompt_tools)

You can use the following tools:
1. `search_tool`: Search on DuckDuckGo navigator using `q` as input
2. `final_answer_tool`: Return a natural language answer to user using the `input text`.
You must provide the maximum of possible context and specify the source of the information.


In [20]:
llm_answer = ollama.chat(model = llm_opensource, messages = [{"role":"system", "content":prompt+"\n"+prompt_tools}, {"role":"user", "content":q}], format = "json")

llm_answer["message"]["content"]

'{"name": "search_tool", "parameters": {"q": "latest premier league winners list"}}'

In [21]:
content = llm_answer["message"]["content"]
data = json.loads(content)

parameters = data.get("parameters", {})
print("Keys in 'parameters':", parameters.keys())

# try to get 'q' value or the input text
tool_input = parameters.get('q') or parameters.get('input text')
if tool_input is None:
    raise KeyError("No one of the keys 'q' or 'input text' were found in 'parameter'.")

print("Value of the tool_input:", tool_input)

Keys in 'parameters': dict_keys(['q'])
Value of the tool_input: latest premier league winners list


In [25]:
# Testing the tool with the context
context = search_tool.invoke(tool_input)
print("Context:\n", context)

Context:
 The English football champions are the annualwinnersof the top-tier competition in the English footballleaguesystem. This is the page for thePremierLeague, with an overview of fixtures, tables, dates, squads, market values, statistics and history.Chelsea ConferenceLeaguewinner2025. View thelateststandings in thePremierLeaguetable, along with form guides and season archives, on the official website of thePremierLeague. EnglishPremierLeagueWinnersList. The English footballleaguewas the first fully-professional football competition in the world. Here are thewinnersfrom each season in the top level footballleaguein England, since 1888. View all match results from thePremierLeague2025/2026 season, including dates and kick off times with GOAL.


## Creating the Agent Structure
On LangGraph, an agent can be developed as a class. This is useful to encapsulate the agent logic, including the nodes definition, trasitions and the memory use and tool.

When defining an agent as a class, you can:
- Structure better the source, spliting the logic and graph execution.
- Storing internal states and configure the decision flow in a more organized way.
- Facilitate the reuse and the scalability of the agent.

In [26]:
# AgentClass
class MyAgent(BaseModel):
    tool_name: str # Tool name used by the agent
    tool_input: dict # Input provided by the agent (parameters)
    tool_output: str | None = None  # Output result
    
    # Method to crate the instance from the model answer (LLM)
    @classmethod
    def from_llm(cls, res: dict):  
        try:
            out = json.loads(res["message"]["content"])
            return cls(tool_name=out["name"], tool_input=out["parameters"])
        except Exception as e:
            print(f"Ollama error:\n{res}\n")
            raise e

In [27]:
# Testing the agent
my_agent = MyAgent.from_llm(llm_answer)
print("Initial format:\n", llm_answer["message"]["content"], "\nAgent format:")
my_agent

Initial format:
 {"name": "search_tool", "parameters": {"q": "latest premier league winners list"}} 
Agent format:


MyAgent(tool_name='search_tool', tool_input={'q': 'latest premier league winners list'}, tool_output=None)

In [28]:
MyAgent(tool_name = "dsa_toosearch_tooll_busca", 
         tool_input = {'q':'Who are the top 10 latest winners of Premier League?'}, 
         tool_output = str( search_tool.invoke(tool_input)) )

MyAgent(tool_name='dsa_toosearch_tooll_busca', tool_input={'q': 'Who are the top 10 latest winners of Premier League?'}, tool_output="EPLwinnerslist1992 to 2025. See all 34 champions, most titles by team, records (100-point season, unbeaten), and dominant eras explained. Get a sneak peek atPremierLeague'srecent champions! Discover the previous seasonswinnersand their key stats, points and goals. EnglishPremierLeagueWinnersList: The EnglishPremierLeagueis one of the most prestigious and popular footballleaguesin the world. It is the highest level of professional football in England. Check out the completelistof EnglishPremierLeaguewinnersdating back to 1888. Here is thelistof all thePremierLeaguewinnersstarting from 1888 to present day. Liverpool & Manchester United have dominated theleague.")

## Setting the System Memory

On LangGraph, the memory refers the ability to store and retrieve informations through the conversation flow. This allows that the agent keep the context among the interactions, and adjust its answers based on previous interactions. The memory can be implemented in different ways, such as storing variables on the graph state or integrating external storage solutions.

In [29]:
# Function to generate contextual memory in th Agent 
def agent_memory(lst_res: list[MyAgent], user_q: str) -> list:
    memory = []
    
    # Iterate about the previous answers that having a valid output
    for res in [res for res in lst_res if res.tool_output is not None]:
        
        memory.extend([
            {"role": "assistant", "content": json.dumps({"name": res.tool_name, "parameters": res.tool_input})},
            {"role": "user", "content": res.tool_output}
        ])
    
    # If the memory is not empty, add a reminder of the original answer 
    if memory:
        memory += [{"role": "user", "content": (f'''
                This is only a reminder that my original query was `{user_q}`.
                Responda apenas à pergunta original e nada mais, mas use as informações que lhe dei.
                Answer only the original question and nothing else, but use the informations I gave you.
                Provide the maximum of possible informations when you use the tool `final_answer`.
                ''')}]

    return memory

In [30]:
chat_history = [{"role": "user", "content": "Hi, how are you doing?"},
                {"role": "assistant", "content": "I'm great, thanks!"},
                {"role": "user", "content": "I have a question"},
                {"role": "assistant", "content": "Yes, what would you like to know?"}]

In [31]:
def execute_agent(prompt: str, 
                       dic_tools: dict, 
                       user_q: str, 
                       chat_history: list[dict], 
                       lst_res: list[MyAgent]) -> MyAgent:
    
    # Initializing the memory with the previous answers from the agent
    memory = agent_memory(lst_res = lst_res, user_q = user_q)
    
    if memory:
        tools_used = [res.tool_name for res in lst_res]
        if len(tools_used) >= len(dic_tools.keys()):
            memory[-1]["content"] = "Now you must to use the tool `final_answer_tool`."

    str_tools = "\n".join([
        f"{n+1}. `{v.name}`: {v.description}" 
        for n, v in enumerate(dic_tools.values())
    ])
    
    prompt_tools = f"You can use the following tools:\n{str_tools}"
        
    # Build the full messages list (prompt + history + memory)
    messages = [
        {"role": "system", "content": prompt + "\n" + prompt_tools},
        *chat_history,
        {"role": "user", "content": user_q},
        *memory
    ]
    
    llm_res = ollama.chat(model = llm_opensource, 
                          messages = messages, 
                          format = "json")
    
    return MyAgent.from_llm(llm_res)

In [33]:
# Testng
agent_res = execute_agent(prompt = prompt, 
                               dic_tools = dic_tools, 
                               user_q = q, 
                               chat_history = chat_history, 
                               lst_res = [])
print("Agent answer:", agent_res)

Agent answer: tool_name='search_tool' tool_input={'q': '{"premierleague.com", "top-10-winners"}'} tool_output=None


## Creating the Graph Workflow Components

The LangGraph graph workflow is a structure that defines how the interactions flow between the graph nodes. It represents the agent logic as in a directed graph, where each node can execute a function and each edge defines possibles transictions to the next steps. 

In [34]:
# State Class
class State(typing.TypedDict):
    user_q: str
    chat_history: list 
    lst_res: list[MyAgent] # Previous answers

    output: dict

In [35]:
# Testando
state = State({"user_q":q, "chat_history":chat_history, "lst_res":[agent_res], "output":{}})
state

{'user_q': 'Who are the top 10 latest winners of Premier League?',
 'chat_history': [{'role': 'user', 'content': 'Hi, how are you doing?'},
  {'role': 'assistant', 'content': "I'm great, thanks!"},
  {'role': 'user', 'content': 'I have a question'},
  {'role': 'assistant', 'content': 'Yes, what would you like to know?'}],
 'lst_res': [MyAgent(tool_name='search_tool', tool_input={'q': '{"premierleague.com", "top-10-winners"}'}, tool_output=None)],
 'output': {}}

## Node
On LangGraph, a node is a processing unit within the Graph Workflow. Each node executes a specific function and can modify the state before passing the execution to another node.

In [36]:
def agent_node(state):
    
    print("--- Agent Node ---")
    
    agent_res = execute_agent(
        prompt = prompt, 
        dic_tools = {k: v for k, v in dic_tools.items() if k in ["search_tool", "final_answer_tool"]},
        user_q = state["user_q"], 
        chat_history = state["chat_history"], 
        lst_res = state["lst_res"]
    )
    
    print(agent_res)
    
    return {"lst_res": [agent_res]}

In [37]:
# Testing
agent_node(state)

--- Agent Node ---
tool_name='search_tool' tool_input={'q': 'Premier League latest winners'} tool_output=None


{'lst_res': [MyAgent(tool_name='search_tool', tool_input={'q': 'Premier League latest winners'}, tool_output=None)]}